# 05 – Controlled input changes (metamorphic probes)

So far we *observed* which test examples fail. Here we *intervene*: take test inputs, change them in a controlled way
where we know exactly what the correct answer becomes, run the model again, and check whether it is still correct.

**How it works (the recipe)**
1. Pick examples: a random sample of test inputs the model solved **correctly**, plus a sample it got **wrong**.
2. Apply each change below. Each change comes with the new expected answer.
3. **Verify the change itself** with the equivalence check (`nr.verdict`): the changed input must really mean what we
   claim. Examples where a change doesn't apply (e.g. only one term to reorder) or would exceed 98 characters are skipped.
4. Run the model on all changed inputs and score them against the new expected answers.
5. Report per change: *of the originally correct examples, how many stay correct?* and *of the originally wrong ones,
   how many become correct?* A model that understands the math should be unaffected by all of these.

| change | input | expected answer |
|---|---|---|
| control (no change) | unchanged | unchanged – must give 100%, checks the pipeline |
| reorder terms | `a - b + c` → `c + a - b` | unchanged |
| swap operands | `x&~y` → `~y&x` (innermost `&`, `\|`, `^`) | unchanged |
| swap x and y | every `x`↔`y` | `x`↔`y` swapped |
| rename x → c | `x` becomes `c` (inputs using only x, y) | renamed the same way; `c` with `y` occurs in training |
| rename x, y → a, b | (inputs using only x, y) | renamed; the pair `a`,`b` never occurs together in training |
| add cancelling terms | insert `+(x&y)-(x&y)` at a random position | unchanged |
| add zero identity | insert `+(x^y)-(x\|y)+(x&y)` (= 0) | unchanged |
| answer as input | the simplified answer itself | unchanged (should be returned as is) |

Running the model on all probes takes a few minutes on a laptop CPU; results are cached in
`probe_results.csv` (delete it to recompute). Set `FULL = True` in the first code cell to run the changes on **all**
test inputs instead of a sample (one-time, ~30-45 min; cached in `probe_results_full.csv`) – notebook `06` uses that
to colour every input by how robust it is.

In [ ]:
import random
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "predictions_test.csv").exists())
sys.path.insert(0, str(ROOT / "src"))
import neureduce as nr

FIG = Path.cwd() / "figures"
FIG.mkdir(exist_ok=True)
# FULL = True runs every change on ALL test inputs (one-time, roughly 30-45 min on a laptop CPU).
# Notebook 06 uses those results to colour the projections by robustness; the analysis below works either way.
FULL = False
CACHE = Path.cwd() / ("probe_results_full.csv" if FULL else "probe_results.csv")

N_CORRECT, N_WRONG, SEED = 1000, 500, 0
test = pd.read_csv(ROOT / "data" / "predictions_test.csv")
if FULL:
    correct, wrong = test[test.correct], test[test.verdict == "wrong"]
else:
    correct = test[test.correct].sample(N_CORRECT, random_state=SEED)
    wrong = test[~test.correct & (test.verdict == "wrong")].sample(N_WRONG, random_state=SEED)
base = pd.concat([correct.assign(orig="correct"), wrong.assign(orig="wrong")])
print(base.orig.value_counts().to_dict())

## 1. The changes

In [ ]:
def equivalent(a, b):
    return nr.verdict(a, b) in ("exact", "equivalent")


def split_terms(expr):
    """Top-level additive terms with their sign: '-2*(x&y)+z' -> ['-2*(x&y)', '+z']."""
    terms, cur, depth = [], "", 0
    for i, ch in enumerate(expr):
        if ch in "+-" and depth == 0 and i > 0 and expr[i - 1] not in "+-*&|^~(":
            terms.append(cur)
            cur = ch
        else:
            cur += ch
        depth += (ch == "(") - (ch == ")")
    return terms + [cur]


def join_terms(terms):
    terms = [t if t[0] in "+-" else "+" + t for t in terms]
    out = "".join(terms)
    return out[1:] if out.startswith("+") else out


def rename(expr, mapping):
    return expr.translate(str.maketrans(mapping))


def two_vars(expr):
    v = sorted(set(re.findall(r"[a-z]", expr)))
    return v[:2] if len(v) >= 2 else None


def insert_block(expr, block_terms, rng):
    terms = split_terms(expr)
    pos = rng.randint(0, len(terms))
    return join_terms(terms[:pos] + block_terms + terms[pos:])


# each probe: (src, trg, rng) -> (new_src, new_expected) or None if it doesn't apply
def p_control(s, t, rng):
    return s, t

def p_reorder(s, t, rng):
    terms = split_terms(s)
    if len(terms) < 2:
        return None
    order = list(range(len(terms)))
    while order == sorted(order):
        rng.shuffle(order)
    return join_terms([terms[i] for i in order]), t

def p_swap_operands(s, t, rng):
    new = re.sub(r"(~?[a-z])([&|^])(~?[a-z])", r"\3\2\1", s)
    return (new, t) if new != s else None

def p_swap_xy(s, t, rng):
    if not {"x", "y"} & set(s):
        return None
    m = {"x": "y", "y": "x"}
    return rename(s, m), rename(t, m)

def p_rename_c(s, t, rng):
    if not set(re.findall(r"[a-z]", s)) <= {"x", "y"} or "x" not in s:
        return None
    return rename(s, {"x": "c"}), rename(t, {"x": "c"})

def p_rename_ab(s, t, rng):
    if not set(re.findall(r"[a-z]", s)) <= {"x", "y"}:
        return None
    m = {"x": "a", "y": "b"}
    return rename(s, m), rename(t, m)

def p_cancel(s, t, rng):
    v = two_vars(s)
    if not v:
        return None
    u, w = v
    return insert_block(s, [f"+({u}&{w})", f"-({u}&{w})"], rng), t

def p_zero_identity(s, t, rng):
    v = two_vars(s)
    if not v:
        return None
    u, w = v
    return insert_block(s, [f"+({u}^{w})", f"-({u}|{w})", f"+({u}&{w})"], rng), t

def p_answer_as_input(s, t, rng):
    return t, t


PROBES = {
    "control (no change)": p_control,
    "reorder terms": p_reorder,
    "swap operands": p_swap_operands,
    "swap x and y": p_swap_xy,
    "rename x -> c": p_rename_c,
    "rename x,y -> a,b": p_rename_ab,
    "add cancelling terms": p_cancel,
    "add zero identity": p_zero_identity,
    "answer as input": p_answer_as_input,
}

# quick look at what each change does
s, t = "2*(x&~y)-(x|y)+(~x^y)", "x"
for name, f in PROBES.items():
    print(f"{name:22} {f(s, t, random.Random(1))}")

In [ ]:
rows, skipped = [], {}
for name, f in PROBES.items():
    rng = random.Random(SEED)
    for r in base.itertuples():
        out = f(r.src, r.trg, rng)
        # skip if the change doesn't apply, would be too long, or (safety net) changed the meaning
        if out is None or len(out[0]) > nr.MAX_POSITIONS - 2 or not equivalent(out[0], out[1]):
            skipped[name] = skipped.get(name, 0) + 1
            continue
        rows.append({"probe": name, "row": r.row, "orig": r.orig, "orig_pred": r.pred, "src": r.src,
                     "trg": r.trg, "new_src": out[0], "new_trg": out[1],
                     "trg_n_terms": r.trg_n_terms, "trg_len": r.trg_len, "trg_n_vars": r.trg_n_vars})
probes = pd.DataFrame(rows)
print("probe inputs:", len(probes))
print("skipped (not applicable / too long):", skipped)

## 2. Run the model on the changed inputs

In [ ]:
if CACHE.exists():
    res = pd.read_csv(CACHE)
    print(f"loaded {len(res):,} cached results from {CACHE.name}")
else:
    model, src_vocab, trg_vocab, *_ = nr.load(ROOT / "NeuReduce-original")
    out = nr.predict(model, probes.new_src.tolist(), src_vocab, trg_vocab)
    res = probes.assign(new_pred=[o["pred"] for o in out], new_min_tok_prob=[o["min_tok_prob"] for o in out])
    res["new_verdict"] = [nr.verdict(p, t) for p, t in zip(res.new_pred, res.new_trg)]
    res.to_csv(CACHE, index=False)
res["new_correct"] = res.new_verdict.isin(["exact", "equivalent"])

## 3. Results

Left: of the examples the model originally solved, how many does it **still** solve after the change?
Right: of the examples it originally got wrong, how many does it **now** solve? (A nonzero right side means its errors
depend on how the input is written, not only on what it means.)

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return centre - half, centre + half

summary = res.groupby(["probe", "orig"]).new_correct.agg(n="size", k="sum").reset_index()
summary["rate"] = summary.k / summary.n
summary["lo"], summary["hi"] = wilson(summary.k, summary.n)
order = list(PROBES)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, orig, color, title in [(axes[0], "correct", "tab:green", "originally correct: still correct"),
                               (axes[1], "wrong", "tab:red", "originally wrong: now correct")]:
    s = summary[summary.orig == orig].set_index("probe").reindex(order)
    y = np.arange(len(order))
    ax.barh(y, s.rate, color=color, alpha=0.8)
    ax.errorbar(s.rate, y, xerr=[s.rate - s.lo, s.hi - s.rate], fmt="none", ecolor="black", capsize=2, lw=0.8)
    for yy, r, hi, n in zip(y, s.rate, s.hi, s.n):
        if not np.isnan(r):
            ax.text(hi + 0.02, yy, f"{r:.0%}  (n={int(n)})", va="center", fontsize=8)
    ax.set_xlim(0, 1.15)
    ax.set_title(title, fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_yticks(np.arange(len(order)), order)
axes[0].invert_yaxis()
fig.suptitle("How does the model react to changes that keep the math the same?")
fig.tight_layout()
fig.savefig(FIG / "probes_overview.png", dpi=200)
summary.pivot(index="probe", columns="orig", values=["rate", "n"]).reindex(order).round(3)

### Does it give the *same* answer, or just *a* correct one?

For originally correct examples that stay correct: is the output literally the original answer (renamed where the
variables were renamed)? A low share means the model rewrites the answer in a different but equivalent form.

In [ ]:
VAR_MAPS = {"swap x and y": {"x": "y", "y": "x"}, "rename x -> c": {"x": "c"}, "rename x,y -> a,b": {"x": "a", "y": "b"}}
ok = res[(res.orig == "correct") & res.new_correct].copy()
ok["same_answer"] = [p == rename(o, VAR_MAPS.get(name, {})) for p, o, name in zip(ok.new_pred, ok.orig_pred, ok.probe)]
ok.groupby("probe").same_answer.mean().reindex(order).round(3).to_frame("same answer string")

## 4. Finding 3 under the microscope: two-variable bitwise answers

Same results, split by the kind of correct answer: short two-variable bitwise answers (`x&y`, `x|~y`, … – the
confusion block from `04`) vs. everything else. If renaming or swapping operands hurts these much more, the model
handles them by surface pattern rather than by meaning.

In [ ]:
res["answer_kind"] = np.where((res.trg_n_terms == 1) & (res.trg_len <= 5) & (res.trg_n_vars == 2),
                              "two-variable bitwise answer", "other answers")
k = res[res.orig == "correct"].groupby(["probe", "answer_kind"]).new_correct.agg(n="size", rate="mean").reset_index()
pv = k.pivot(index="probe", columns="answer_kind", values="rate").reindex(order)
nv = k.pivot(index="probe", columns="answer_kind", values="n").reindex(order)

fig, ax = plt.subplots(figsize=(8, 4.2))
y = np.arange(len(order))
for i, (kind, color) in enumerate([("other answers", "tab:gray"), ("two-variable bitwise answer", "tab:purple")]):
    ax.barh(y + (i - 0.5) * 0.38, pv[kind], height=0.38, color=color, label=kind)
    for yy, r, n in zip(y, pv[kind], nv[kind]):
        if not np.isnan(r):
            ax.text(r + 0.01, yy + (i - 0.5) * 0.38, f"{r:.0%} (n={int(n)})", va="center", fontsize=7)
ax.set_yticks(y, order)
ax.invert_yaxis()
ax.set_xlim(0, 1.2)
ax.set_xlabel("originally correct examples that stay correct")
ax.legend(fontsize=8, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIG / "probes_by_answer_kind.png", dpi=200)
pv.round(3)

## 5. Read examples where a change broke a correct answer

In [ ]:
pd.set_option("display.max_colwidth", 70)
broken = res[(res.orig == "correct") & ~res.new_correct & (res.probe != "control (no change)")]
(broken.sample(frac=1, random_state=0).groupby("probe").head(3)
       .assign(o=lambda d: d.probe.map(order.index)).sort_values("o")
       [["probe", "src", "new_src", "new_trg", "new_pred", "new_verdict"]])

## Writing it up

> *"When we change inputs in ways that don't change their meaning, the model stays correct in X–Y% of cases.
> [Change] hurts most (Z%), and two-variable bitwise answers are [more / not more] fragile. Renaming variables to
> letters it rarely saw together drops accuracy to …, so the model partly relies on the surface form of the input."*